In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as Figure
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('/content/drive/MyDrive/Monetary_Economics.csv')
df.head()

,COUNTRY,MONTH,INF,EXR,INR,INF2,WOP,GEA,GEA2
0,ALGERIA,01/01/2000,2.242235,70.7292,10.0,4.697356,27.224286,-10.028534,0.049734
1,ALGERIA,01/02/2000,2.426637,72.4357,10.0,5.051244,29.362381,-8.681862,0.057402
2,ALGERIA,01/03/2000,1.660758,73.4741,10.0,3.599344,29.892174,5.995629,12.074081
3,ALGERIA,01/04/2000,0.924354,76.0025,10.0,2.286129,25.799000,9.081429,18.217750
4,ALGERIA,01/05/2000,-0.485800,75.2502,10.0,0.625956,28.833478,5.096124,10.289435


In [3]:
df.columns

Index(['COUNTRY', 'MONTH', 'INF', 'EXR', 'INR', 'INF2', 'WOP', 'GEA', 'GEA2 '], dtype='object')

The Columns **INF2** and **GEA2** are dropped since they dont show statistical consistency or the records in the dataset are not compatatible to to how they could be gotten

In [4]:
df = df.drop(['INF2','GEA2 '],axis=1)
df.head()

,COUNTRY,MONTH,INF,EXR,INR,WOP,GEA
0,ALGERIA,01/01/2000,2.242235,70.7292,10.0,27.224286,-10.028534
1,ALGERIA,01/02/2000,2.426637,72.4357,10.0,29.362381,-8.681862
2,ALGERIA,01/03/2000,1.660758,73.4741,10.0,29.892174,5.995629
3,ALGERIA,01/04/2000,0.924354,76.0025,10.0,25.799000,9.081429
4,ALGERIA,01/05/2000,-0.485800,75.2502,10.0,28.833478,5.096124


# Monetary Economics Explained

**CPI Inflation Rate (INF):** How much more expensive everyday items became
compared to the exact same month last year.

**Exchange Rate (EXR):** The amount of local money it takes to buy exactly one US dollar.

**Lending Interest Rate (INR):** The annual percentage fee that local commercial banks charge normal people or businesses to borrow money.

# International Finance Metrics

**Crude Oil Price (WOP):** The cost of one barrel of standard American oil, which acts as the global baseline for fuel prices.

**Index of Global Real Economic Activity (GEA)**: A global scorecard that counts how many physical goods are moving across oceans; a high score means global trade and shipping are booming, while a low score means trade is slowing down.

In [5]:
#Rename the Columns to be more familiar terms
df.rename(columns={
    "COUNTRY": "Country",
    "MONTH": "Date",
    "INF": "CPI Inflation",
    "EXR": "Exchange Rate",
    "INR": " Lending Interest Rate",
    "WOP": "World Oil Price",
    "GEA": "Global Economic Activity"
}, inplace=True)

df.head()

,Country,Date,CPI Inflation,Exchange Rate,Lending Interest Rate,World Oil Price,Global Economic Activity
0,ALGERIA,01/01/2000,2.242235,70.7292,10.0,27.224286,-10.028534
1,ALGERIA,01/02/2000,2.426637,72.4357,10.0,29.362381,-8.681862
2,ALGERIA,01/03/2000,1.660758,73.4741,10.0,29.892174,5.995629
3,ALGERIA,01/04/2000,0.924354,76.0025,10.0,25.799000,9.081429
4,ALGERIA,01/05/2000,-0.485800,75.2502,10.0,28.833478,5.096124


# Feature Engineering

# Time Lag Features
A lag represents a delay in the transmission of an economic shock from its source to its destination.

### 1.Supply Chain Delays

Oil bought on global markets today takes weeks to be shipped, refined, landed at port, and distributed to local petrol stations.

In [11]:
df = df.sort_values(by=['Country','Date']).reset_index(drop=True)

#Added Time Lags
df['wop_lag1']= df.groupby('Country')['World Oil Price'].shift(1)
df['wop_lag2']= df.groupby('Country')['World Oil Price'].shift(2)
df['gea_lag1']= df.groupby('Country')['Global Economic Activity'].shift(1)

# Momentum and Rate Of Change (The Shocks)

In macroeconomic modeling, levels (e.g., oil staying at $80/barrel for six months) represent structural stability. Economies, supply chains, and central banks adapt to steady states. A shock, however, is a sudden, volatile deviation from that stability.

In [12]:
df['wop_pct_change'] = df.groupby('Country')['World Oil Price'].pct_change(1)

# Moving Averages
 Single-month spikes can sometimes be noise. Local economies are heavily influenced by the sustained, medium-term trend of global markets.

 ## Code Warning

WHY NOT TO USE:

```df.groupby('Country')['WOP'].transform(lambda x: x.rolling(3).mean())```

The line above includes the CURRENT month's data in the average calculation.
If used to predict current inflation, it introduces "Data Leakage" (look-ahead bias), effectively letting the model cheat by looking at the present to predict the present.

In [13]:
df['oil_moving_average'] = df.groupby('Country')['World Oil Price'].transform(lambda x: x.shift(1).rolling(3).mean())
df['gea_moving_average'] = df.groupby('Country')['Global Economic Activity'].transform(lambda x: x.shift(1).rolling(3).mean())

# Volatility

Volatility measures the speed and size of price swings. It doesn't care if a price is high or low; it cares how unpredictable the price is.

### Acts as a Risk Gauge

High volatility signals global panic or geopolitical tension. Machine learning algorithms use it as a switch to anticipate sudden economic instability.

In [14]:
df['wop_volatility'] = df.groupby('Country')['World Oil Price'].transform(lambda x: x.shift(1).rolling(3).std())

# Interaction Term

In statistics and machine learning, an interaction term is a feature created by multiplying two (or more) independent variables together.

## How it comes to play

In the dataset, you have Oil Prices (WOP) and the Global Economic Activity Index (GEA).

If  i don't use an interaction term, your model assumes that a $100 oil price always has the exact same impact on an economy. But in the real world, the impact of a high oil price depends heavily on the state of global trade

In [15]:
# Z-Score scaling
wop_scaled = (df['World Oil Price'] - df['World Oil Price'].mean()) / df['World Oil Price'].std()
gea_scaled = (df['Global Economic Activity'] - df['Global Economic Activity'].mean()) / df['Global Economic Activity'].std()

# Create the interaction term
df['Global_Pressure_Index']= wop_scaled * gea_scaled

In [17]:
df = df.dropna().reset_index(drop=True)
df.head()

,Country,Date,CPI Inflation,Exchange Rate,Lending Interest Rate,World Oil Price,Global Economic Activity,wop_lag1,wop_lag2,gea_lag1,wop_pct_change,oil_moving_average,gea_moving_average,wop_volatility,Global_Pressure_Index
0,ALGERIA,01/01/2003,-2.139678,79.6407,8.5,32.911739,9.588997,19.607391,29.482174,-49.558033,0.678537,25.437950,-19.320676,5.174078,-0.013330
1,ALGERIA,01/01/2004,6.209550,71.5743,8.0,34.146364,122.304010,32.911739,19.607391,9.588997,0.037513,27.333768,-12.781499,6.907471,-1.852013
2,ALGERIA,01/01/2005,2.947958,72.7691,8.0,46.991905,106.537310,34.146364,32.911739,122.304010,0.376191,28.888498,27.444991,8.061345,-0.888129
3,ALGERIA,01/01/2006,0.626118,72.7956,8.0,65.232727,35.778229,46.991905,34.146364,106.537310,0.388169,38.016669,79.476772,7.797257,0.031813
4,ALGERIA,01/01/2007,2.373333,71.6075,8.0,54.780000,103.666780,65.232727,46.991905,35.778229,-0.160237,48.790332,88.206516,15.621020,-0.446216


In [18]:
# Save the CSV File
df.to_csv('Monetary_Economics_Cleaned.csv', index=False)